# 현장형 평가 — 장비의 과거로 학습하고 미래에 경보

| 항목 | 내용 |
|---|---|
| 질문 | 현장처럼 **그 시점까지 알 수 있는 정보만으로** 경보를 울리면, 고장을 미리 몇 건이나 잡고 오경보는 얼마나 나는가? |
| 종류 | **지도학습 (분류)** 5종 + 비교용 **비지도** 1종(IsolationForest, 정답은 채점에만 사용) |
| 정답(라벨) | 고장 종류별로 따로: 24시간 안에 그 고장이 나는가 (예=1) |
| 데이터 | `data/processed/step/` (레시피 스텝 구간 = 한 행), 셔터 1(식각 중)만 |
| 분할 | 원본 대회와 같은 **같은 장비의 시간 분할**. 장비마다 앞 75% 기간 = 학습, 뒤 25% = 평가 |

## 1. 방법

**기간 나누기 (장비마다)**

| 구간 | 범위 | 쓰는 곳 |
|---|---|---|
| 학습(fit) | 처음 ~ 학습 기간의 2/3 | 평소 기준(z값) 계산, 모델 학습 |
| 보정(calib) | 학습 기간의 나머지 1/3 | 장비별 경보 기준 정하기 (모델이 보지 않은 구간) |
| 간격 | 1일 | 경계에서 정보가 새지 않게 비움 |
| 평가(test) | 뒤 25% | 채점만 |

**입력 (모두 그 시점까지 알 수 있는 값)**

| 묶음 | 내용 | 개수 |
|---|---|---|
| S. 센서 | 냉각 압력·냉각 유량·억제 전류·챔버 압력의 z값(평균·최소·최대, 기준 = 그 장비의 학습 구간 평소 값, 레시피·스텝별) + 과거 6시간·24시간·7일 이동 평균 + 변화량(24시간 평균 − 7일 평균) | 28 |
| H. 고장 이력 | 직전 고장(종류 무관)·직전 압력 저하·직전 압력 과다·직전 누설 이후 지난 시간, 최근 7일 고장 수 | 5 |

입력은 **S만** 쓴 경우와 **S+H**를 비교한다. 장비 이름·Lot·runnum·회전·누적 카운터는 넣지 않는다.

**경보 규칙**
- 모델 점수가 장비별 기준을 넘으면 경보. 경보 뒤 24시간은 다시 울리지 않는다.
- 장비별 기준: 보정 구간에서 **오경보가 주 1회 이하**가 되는 가장 낮은 점수.
- 경보 뒤 24시간 안에 그 종류 고장이 나면 맞은 경보, 아니면 오경보.

**채점 (평가 구간)**
- **사건**: 같은 종류 고장 기록을 1일 간격으로 묶은 것. 사건의 첫 기록 전 24시간 안에 경보가 있으면 탐지.
- **판단 가능 사건**: 첫 기록 전 24시간 안에 셔터 1 기록이 하나라도 있는 사건(기록이 없으면 원리적으로 못 잡는다).
- **반복 기록**: 사건 안의 두 번째 이후 기록. 따로 탐지율을 본다.
- 오경보는 장비·주 단위로 센다.

**모델**: 로지스틱 회귀, 랜덤포레스트, 엑스트라트리, LightGBM, XGBoost(이상 지도), IsolationForest(비지도, 센서만). XGBoost만 GPU(RTX 2060)로 학습한다. 설치된 LightGBM은 GPU 빌드가 아니고, scikit-learn 모델은 GPU를 지원하지 않는다.

**기준선**: (1) **직후 경보 규칙** — 어떤 고장이든 나면 그때 경보(학습 없음). (2) **무작위** — 같은 수의 경보를 평가 기간에 무작위로 뿌렸을 때의 기대 탐지율(근사)을 모든 행에 함께 적는다.

In [1]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.display import display
warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, cwd.parent) if (p / 'data/processed/step').exists())
PROC = ROOT / 'data/processed'

def show(fig, height=480):
    fig.update_layout(template='plotly_white', height=height, font=dict(family='Malgun Gothic, sans-serif'),
                      margin=dict(l=30, r=30, t=75, b=60))
    display({'application/vnd.plotly.v1+json': json.loads(fig.to_json())}, raw=True)

SENSORS = ['FLOWCOOLPRESSURE', 'FLOWCOOLFLOWRATE', 'ETCHSUPPRESSORCURRENT', 'IONGAUGEPRESSURE']
STATS = ['mean', 'min', 'max']
FAULTS = ['FlowCool Pressure Dropped Below Limit', 'Flowcool Pressure Too High Check Flowcool Pump', 'Flowcool leak']
FSHORT = dict(zip(FAULTS, ['압력 저하', '압력 과다', '누설']))
TTF = {f: 'TTF_' + f for f in FAULTS}
HOUR, DAY, WEEK = 3600, 86400, 7 * 86400
H = DAY            # 라벨·탐지 창 24시간
FA_TARGET = 1.0    # 보정 구간 오경보 목표(주당)
SEED = 0

cols = ['Tool', 'time_start', 'time_end', 'recipe', 'recipe_step', 'shutter', *TTF.values(),
        *[f'{s}_{st}' for s in SENSORS for st in STATS]]
d = pd.concat([pd.read_parquet(f, columns=cols) for f in sorted((PROC / 'step').glob('*.parquet'))], ignore_index=True)
d['Tool'] = d.Tool.astype(str)

faults = pd.read_csv(PROC / 'faults.csv').drop_duplicates()
faults['Tool'] = faults.Tool.map(lambda t: f'{t[2:]}_{t[:2]}')
faults = faults.sort_values(['Tool', 'time']).reset_index(drop=True)

# 장비별 기간 나누기 (전체 구간 기준)
span = d.groupby('Tool').agg(t0=('time_start', 'min'), t1=('time_end', 'max'))
span['cut'] = span.t0 + 0.75 * (span.t1 - span.t0)
span['fit_end'] = span.t0 + (span.cut - DAY - span.t0) * 2 / 3

# 평소 여부(가장 가까운 고장까지 24시간 초과, 종류 무관)
fdist = np.full(len(d), np.inf)
for tool, idx in d.groupby('Tool').indices.items():
    ft = faults.loc[faults.Tool == tool, 'time'].to_numpy()
    ts, te = d.time_start.to_numpy()[idx], d.time_end.to_numpy()[idx]
    j = np.searchsorted(ft, ts); k = np.searchsorted(ft, te)
    prev = np.where(j > 0, ts - ft[np.maximum(j - 1, 0)], np.inf)
    nxt = np.where(k < len(ft), ft[np.minimum(k, len(ft) - 1)] - te, np.inf)
    fdist[idx] = np.minimum(prev, nxt)
d['normal'] = fdist > DAY

d = d[d.shutter == 1].join(span, on='Tool')
d = d.sort_values(['Tool', 'time_start']).reset_index(drop=True)
d['part'] = np.select([d.time_end < d.fit_end, d.time_end < d.cut - DAY, d.time_start >= d.cut],
                      ['fit', 'calib', 'test'], 'gap')
d['_rs'] = d.recipe_step.fillna(-1)
part = d.part.to_numpy()
tool_arr = d.Tool.to_numpy()
t_arr = d.time_start.to_numpy()

tab = d.groupby('part').agg(구간=('Tool', 'size'), 장비=('Tool', 'nunique')).reindex(['fit', 'calib', 'gap', 'test'])
tab['비율(%)'] = (tab.구간 / tab.구간.sum() * 100).round(1)
display(tab)
print(f'평가 기간 합계 {((span.t1 - span.cut).sum() / WEEK):.0f}장비·주')

,구간,장비,비율(%)
part,,,
fit,278002,20,52.8
calib,120259,20,22.9
gap,1236,20,0.2
test,126769,20,24.1


평가 기간 합계 316장비·주


## 2. 입력 만들기 (그 시점까지 알 수 있는 값만)

In [2]:
MEANS = [s + '_mean' for s in SENSORS]
keys = ['Tool', 'recipe', '_rs']
ref = d[(part == 'fit') & d.normal.to_numpy()]
g = ref.groupby(keys)[MEANS]
n = g.size(); mu, sd = g.mean(), g.std()
sd = sd.where((sd > 1e-6) & np.broadcast_to((n >= 10).to_numpy()[:, None], sd.shape))
key = pd.MultiIndex.from_frame(d[keys])
m = mu.reindex(key).to_numpy(); s_ = sd.reindex(key).to_numpy()
F = {}
for k, s in enumerate(SENSORS):
    for st in STATS:
        F[f'{s}_{st}_z'] = np.clip((d[f'{s}_{st}'].to_numpy() - m[:, k]) / s_[:, k], -10, 10)
F = pd.DataFrame(F).astype('float32')

zm = F[[f'{s}_mean_z' for s in SENSORS]].copy()
zm.index = pd.to_datetime(d.time_start, unit='s')
roll = {}
for win in ['6h', '24h', '7d']:
    r = zm.groupby(tool_arr).rolling(win, min_periods=3).mean().reset_index(level=0, drop=True)
    roll[win] = r.reset_index(drop=True)   # d가 장비·시각 순이라 행 순서가 같다
    for c in zm.columns:
        F[f'{c}_r{win}'] = roll[win][c].to_numpy()
for c in zm.columns:
    F[f'{c}_chg'] = F[f'{c}_r24h'] - F[f'{c}_r7d']
S_COLS = list(F.columns)

# 고장 이력: 그 행 시작 시각 이전에 기록된 고장만
CAP = 24 * 365.0
hist = {c: np.full(len(d), CAP, dtype='float32') for c in ['since_any', 'since_drop', 'since_high', 'since_leak']}
hist['n7d_any'] = np.zeros(len(d), dtype='float32')
for tool, idx in d.groupby('Tool').indices.items():
    ts = t_arr[idx]
    fa = faults[faults.Tool == tool]
    for col, sub in [('since_any', fa), ('since_drop', fa[fa.fault_name == FAULTS[0]]),
                     ('since_high', fa[fa.fault_name == FAULTS[1]]), ('since_leak', fa[fa.fault_name == FAULTS[2]])]:
        ft = sub.time.to_numpy()
        if len(ft) == 0:
            continue
        j = np.searchsorted(ft, ts, 'left')
        hist[col][idx] = np.where(j > 0, np.minimum((ts - ft[np.maximum(j - 1, 0)]) / HOUR, CAP), CAP)
    ft = fa.time.to_numpy()
    hist['n7d_any'][idx] = np.searchsorted(ft, ts, 'left') - np.searchsorted(ft, ts - WEEK, 'left')
for c, v in hist.items():
    F[c] = v
H_COLS = list(hist)
FEATSETS = {'S (센서)': S_COLS, 'S+H (센서+고장 이력)': S_COLS + H_COLS}
print(f'입력: S {len(S_COLS)}개, H {len(H_COLS)}개 · 냉각 압력 z 있음(평가 구간) {F.loc[part == "test", "FLOWCOOLPRESSURE_mean_z"].notna().mean():.1%}')

입력: S 28개, H 5개 · 냉각 압력 z 있음(평가 구간) 97.6%


## 3. 경보·채점 함수

In [3]:
def gen_alarms(t, s, th):
    # 점수가 기준을 넘는 시각들 중, 직전 경보 뒤 24시간이 지난 것만 경보
    ex = t[s > th]
    out, i = [], 0
    while i < len(ex):
        out.append(ex[i]); i = np.searchsorted(ex, ex[i] + DAY, 'left')
    return np.asarray(out, dtype=float)

def false_alarms(alarms, ft):
    if len(alarms) == 0:
        return 0
    if len(ft) == 0:
        return len(alarms)
    j =np.searchsorted(ft, alarms, 'right')   # 경보 이후 첫 고장
    nxt = np.where(j < len(ft), ft[np.minimum(j, len(ft) - 1)], np.inf)
    return int(((nxt - alarms) > H).sum())

def events_of(ft):
    # 같은 종류 기록을 1일 간격으로 묶은 사건의 시작 시각, 반복 기록 시각
    if len(ft) == 0:
        return np.array([]), np.array([])
    new = np.r_[True, np.diff(ft) > DAY]
    return ft[new], ft[~new]

def calib_threshold(t, s, ft, weeks):
    if len(s) == 0 or weeks <= 0:
        return np.inf
    for q in np.r_[np.linspace(0.80, 0.99, 20), 1 - np.logspace(-2.2, -5, 15)]:
        th = np.quantile(s, q)
        if false_alarms(gen_alarms(t, s, th), ft) / weeks <= FA_TARGET:
            return th
    return np.inf

def evaluate(alarm_dict, fault):
    # alarm_dict: 장비 -> 평가 구간 경보 시각 배열
    ev_n = ev_ok = ev_hit = rep_n = rep_hit = n_al = n_fa = 0
    leads, rand_p, weeks = [], [], 0.0
    for tool, sp in span.iterrows():
        a = np.sort(alarm_dict.get(tool, np.array([])))
        ft_all = faults.loc[(faults.Tool == tool) & (faults.fault_name == fault), 'time'].to_numpy()
        starts, reps = events_of(ft_all)
        starts = starts[starts >= sp.cut + H]; reps = reps[reps >= sp.cut + H]
        T = sp.t1 - sp.cut; weeks += T / WEEK
        n_al += len(a); n_fa += false_alarms(a, ft_all)
        tt = t_arr[(tool_arr == tool) & (part == 'test')]
        for st in starts:
            ev_n += 1
            ev_ok += int(np.searchsorted(tt, st - H) < np.searchsorted(tt, st))
            w = a[(a >= st - H) & (a < st)]
            if len(w):
                ev_hit += 1; leads.append((st - w[0]) / HOUR)
            rand_p.append(1 - (1 - min(H / T, 1)) ** len(a))
        for r in reps:
            rep_n += 1; rep_hit += int(((a >= r - H) & (a < r)).any())
    return {'사건': ev_n, '판단 가능 사건': ev_ok, '탐지 사건': ev_hit,
            '탐지율(%)': round(ev_hit / max(ev_n, 1) * 100, 1),
            '판단 가능 중 탐지율(%)': round(ev_hit / max(ev_ok, 1) * 100, 1),
            '무작위 기대 탐지율(%)': round(np.mean(rand_p) * 100, 1) if rand_p else np.nan,
            '선행 시간 중앙값(h)': round(np.median(leads), 1) if leads else np.nan,
            '경보 수': n_al, '오경보/주': round(n_fa / weeks, 2),
            '정밀도(%)': round((n_al - n_fa) / max(n_al, 1) * 100, 1),
            '반복 기록 탐지율(%)': round(rep_hit / max(rep_n, 1) * 100, 1)}

## 4. 학습과 평가

In [4]:
def make_models(pos_w):
    return {
        '로지스틱 회귀': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000)),
        '랜덤포레스트': RandomForestClassifier(n_estimators=200, min_samples_leaf=100, max_features='sqrt',
                                          class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
        '엑스트라트리': ExtraTreesClassifier(n_estimators=200, min_samples_leaf=100, max_features='sqrt',
                                        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED),
        'LightGBM': lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=200,
                                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, is_unbalance=True,
                                       random_state=SEED, verbose=-1),
        'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                                 tree_method='hist', device='cuda', scale_pos_weight=pos_w, n_jobs=-1, random_state=SEED, verbosity=0),
    }
NEEDS_FILL = {'로지스틱 회귀', '랜덤포레스트', '엑스트라트리'}   # 결측(z 없음)을 0(=평소)으로 채움

is_fit, is_cal, is_te = part == 'fit', part == 'calib', part == 'test'
cal_weeks = ((span.cut - DAY - span.fit_end) / WEEK).to_dict()

def run(name_model, featset, fault, s_cal, s_te, rows):
    ft_by_tool = {t: faults.loc[(faults.Tool == t) & (faults.fault_name == fault), 'time'].to_numpy() for t in span.index}
    alarms = {}
    for tool in span.index:
        ic, it = cal_idx[tool], te_idx[tool]
        th = calib_threshold(t_arr[ic], s_cal[ic], ft_by_tool[tool], cal_weeks[tool])
        alarms[tool] = gen_alarms(t_arr[it], s_te[it], th)
    y_te = (d[TTF[fault]].to_numpy()[is_te] <= H)
    auc_te = roc_auc_score(y_te, s_te[is_te]) if 0 < y_te.sum() < len(y_te) else np.nan
    rows.append({'고장': FSHORT[fault], '입력': featset, '모델': name_model, '행 AUC(평가)': round(auc_te, 3),
                 **evaluate(alarms, fault)})

cal_idx = {t: np.flatnonzero(is_cal & (tool_arr == t)) for t in span.index}
te_idx = {t: np.flatnonzero(is_te & (tool_arr == t)) for t in span.index}

rows, timing = [], []
# 비지도: IsolationForest (센서만, 학습 구간 평소 행으로 학습, 고장 종류와 무관하게 한 번)
t0 = time.time()
Xs = F[S_COLS].fillna(0).to_numpy()
fit_norm = np.flatnonzero(is_fit & d.normal.to_numpy())
iso = IsolationForest(n_estimators=200, max_samples=4096, random_state=SEED, n_jobs=-1)
iso.fit(Xs[np.random.default_rng(SEED).choice(fit_norm, min(200_000, len(fit_norm)), replace=False)])
s_iso = -iso.score_samples(Xs)
for fault in FAULTS:
    run('IsolationForest (비지도)', 'S (센서)', fault, s_iso, s_iso, rows)
timing.append(('IsolationForest', round(time.time() - t0)))
print('IsolationForest 완료', flush=True)

for fault in FAULTS:
    ttf = d[TTF[fault]].to_numpy()
    censored = ((d.t1.to_numpy() - d.time_end.to_numpy()) < H) & np.isnan(ttf)
    y = (ttf <= H).astype(int)
    tr = np.flatnonzero(is_fit & ~censored)
    pos_w = (len(tr) - y[tr].sum()) / max(y[tr].sum(), 1)
    for featset, fcols in FEATSETS.items():
        X = F[fcols]
        Xf = X.fillna(0)
        for name, model in make_models(pos_w).items():
            t0 = time.time()
            XX = Xf if name in NEEDS_FILL else X
            model.fit(XX.iloc[tr], y[tr])
            score = model.predict_proba(XX)[:, 1]
            run(name, featset, fault, score, score, rows)
            timing.append((f'{FSHORT[fault]} · {featset} · {name}', round(time.time() - t0)))
        print(f'{FSHORT[fault]} · {featset} 완료', flush=True)

# 기준선: 직후 경보 규칙 (어떤 고장이든 기록되면 그 시각에 경보)
for fault in FAULTS:
    alarms = {}
    for tool, sp in span.iterrows():
        ft = faults.loc[faults.Tool == tool, 'time'].to_numpy()
        ft = ft[ft >= sp.cut]
        alarms[tool] = gen_alarms(ft, np.ones(len(ft)), 0)
    rows.append({'고장': FSHORT[fault], '입력': '-', '모델': '기준선: 직후 경보 규칙', '행 AUC(평가)': np.nan, **evaluate(alarms, fault)})
res = pd.DataFrame(rows)
print('총 학습 시간(초):', sum(t for _, t in timing))

IsolationForest 완료


압력 저하 · S (센서) 완료


압력 저하 · S+H (센서+고장 이력) 완료


압력 과다 · S (센서) 완료


압력 과다 · S+H (센서+고장 이력) 완료


누설 · S (센서) 완료


누설 · S+H (센서+고장 이력) 완료


총 학습 시간(초): 290


### 4-1. 결과표

- **탐지율**: 평가 구간 사건 중, 첫 기록 전 24시간 안에 경보가 울린 비율. **무작위 기대 탐지율**은 같은 수의 경보를 무작위로 뿌렸을 때의 근사값이다. 탐지율이 이보다 높아야 의미가 있다.
- **오경보/주**: 장비 한 대가 일주일에 받는 오경보 수(평균). 경보 기준은 보정 구간에서 주 1회 이하가 되게 정했다.
- **반복 기록 탐지율**: 같은 사건 안 두 번째 이후 기록을 미리 잡은 비율(첫 고장보다 쉬운 문제).

In [5]:
order = ['기준선: 직후 경보 규칙', 'IsolationForest (비지도)', '로지스틱 회귀', '랜덤포레스트', '엑스트라트리', 'LightGBM', 'XGBoost']
res['_o'] = res.모델.map(order.index)
for fault in FAULTS:
    r = res[res.고장 == FSHORT[fault]].sort_values(['_o', '입력']).drop(columns=['고장', '_o'])
    print(f'■ {FSHORT[fault]} — 사건 {r.사건.iloc[0]}건 (판단 가능 {r["판단 가능 사건"].iloc[0]}건)')
    display(r.drop(columns=['사건', '판단 가능 사건']).reset_index(drop=True))

■ 압력 저하 — 사건 67건 (판단 가능 67건)


,입력,모델,행 AUC(평가),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,-,기준선: 직후 경보 규칙,NaN,4,6.0,6.0,12.7,7.3,154,0.36,25.3,78.5
1,S (센서),IsolationForest (비지도),0.501,22,32.8,32.8,32.6,8.6,874,2.67,3.7,40.0
2,S (센서),로지스틱 회귀,0.556,15,22.4,22.4,26.1,8.8,570,1.73,4.2,26.2
3,S+H (센서+고장 이력),로지스틱 회귀,0.715,24,35.8,35.8,26.8,11.0,626,1.86,6.4,47.7
4,S (센서),랜덤포레스트,0.562,9,13.4,13.4,18.9,4.3,472,1.43,4.0,23.1
5,S+H (센서+고장 이력),랜덤포레스트,0.808,18,26.9,26.9,22.1,13.0,643,1.91,6.4,61.5
6,S (센서),엑스트라트리,0.585,10,14.9,14.9,14.8,15.9,393,1.20,3.6,16.9
7,S+H (센서+고장 이력),엑스트라트리,0.815,15,22.4,22.4,16.6,5.3,498,1.50,5.0,35.4
8,S (센서),LightGBM,0.540,11,16.4,16.4,23.1,13.2,495,1.50,4.2,27.7
9,S+H (센서+고장 이력),LightGBM,0.728,18,26.9,26.9,21.7,7.1,442,1.30,6.8,38.5


■ 압력 과다 — 사건 68건 (판단 가능 47건)


,입력,모델,행 AUC(평가),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,-,기준선: 직후 경보 규칙,NaN,4,5.9,8.5,12.6,13.7,154,0.46,6.5,90.0
1,S (센서),IsolationForest (비지도),0.741,24,35.3,51.1,26.4,13.8,819,2.51,3.3,60.0
2,S (센서),로지스틱 회귀,0.307,13,19.1,27.7,24.3,12.7,625,1.94,2.2,40.0
3,S+H (센서+고장 이력),로지스틱 회귀,0.367,8,11.8,17.0,13.3,13.9,616,1.92,1.6,10.0
4,S (센서),랜덤포레스트,0.333,4,5.9,8.5,12.5,2.7,432,1.35,1.2,30.0
5,S+H (센서+고장 이력),랜덤포레스트,0.567,4,5.9,8.5,10.1,17.7,549,1.72,1.1,10.0
6,S (센서),엑스트라트리,0.300,5,7.4,10.6,14.3,14.7,323,1.01,1.5,30.0
7,S+H (센서+고장 이력),엑스트라트리,0.491,5,7.4,10.6,12.2,6.2,727,2.28,0.8,30.0
8,S (센서),LightGBM,0.446,6,8.8,12.8,8.2,5.8,320,0.99,2.2,10.0
9,S+H (센서+고장 이력),LightGBM,0.520,10,14.7,21.3,14.1,7.4,479,1.48,2.5,50.0


■ 누설 — 사건 15건 (판단 가능 13건)


,입력,모델,행 AUC(평가),탐지 사건,탐지율(%),판단 가능 중 탐지율(%),무작위 기대 탐지율(%),선행 시간 중앙값(h),경보 수,오경보/주,정밀도(%),반복 기록 탐지율(%)
0,-,기준선: 직후 경보 규칙,NaN,7,46.7,53.8,9.6,1.6,154,0.45,8.4,88.2
1,S (센서),IsolationForest (비지도),0.347,3,20.0,23.1,21.5,12.5,822,2.59,0.5,52.9
2,S (센서),로지스틱 회귀,0.492,2,13.3,15.4,13.3,5.8,521,1.64,0.4,11.8
3,S+H (센서+고장 이력),로지스틱 회귀,0.737,3,20.0,23.1,21.9,10.1,512,1.60,1.2,47.1
4,S (센서),랜덤포레스트,0.420,1,6.7,7.7,10.4,1.6,587,1.85,0.3,5.9
5,S+H (센서+고장 이력),랜덤포레스트,0.797,4,26.7,30.8,16.0,5.5,547,1.70,1.6,58.8
6,S (센서),엑스트라트리,0.541,4,26.7,30.8,15.5,1.3,414,1.30,1.0,0.0
7,S+H (센서+고장 이력),엑스트라트리,0.847,7,46.7,53.8,17.8,5.1,574,1.79,1.6,35.3
8,S (센서),LightGBM,0.469,1,6.7,7.7,13.0,2.2,400,1.26,0.2,23.5
9,S+H (센서+고장 이력),LightGBM,0.787,4,26.7,30.8,17.2,4.7,296,0.91,2.7,47.1


### 4-2. 탐지율과 오경보

In [6]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[FSHORT[f] for f in FAULTS], shared_yaxes=True)
colors = {'S (센서)': '#2a78d6', 'S+H (센서+고장 이력)': '#eb6834', '-': '#444'}
for j, fault in enumerate(FAULTS, 1):
    r = res[res.고장 == FSHORT[fault]]
    for fs, grp in r.groupby('입력'):
        fig.add_scatter(x=grp['오경보/주'], y=grp['탐지율(%)'], mode='markers+text', text=grp.모델.str.replace('기준선: ', ''),
                        textposition='top center', textfont=dict(size=9), marker=dict(size=10, color=colors[fs]),
                        name=fs if fs != '-' else '기준선', legendgroup=fs, showlegend=j == 1, row=1, col=j)
    fig.add_scatter(x=r['오경보/주'], y=r['무작위 기대 탐지율(%)'], mode='markers', marker=dict(symbol='x', size=7, color='#9aa3ab'),
                    name='같은 경보 수로 무작위', legendgroup='rand', showlegend=j == 1, row=1, col=j)
fig.update_xaxes(title_text='오경보/주 (장비 1대)')
fig.update_yaxes(title_text='사건 탐지율(%)', col=1)
fig.update_layout(title='그림 4-2. 모델별 사건 탐지율과 오경보 (평가 구간 = 장비별 뒤 25%)', legend=dict(orientation='h', y=-0.2))
show(fig, 560)

## 5. 해석과 한계

**결론: 현장과 같은 조건에서는 첫 고장을 미리 잡는 능력이 약하다.** 가장 나은 조합도 사건의 약 1/3을 잡는 대신 장비당 주 2회 정도 오경보가 나고, 경보 중 맞는 비율(정밀도)은 10%가 안 된다.

| 고장 | 가장 나은 조합 | 탐지율 | 같은 경보 수로 무작위 | 오경보/주 | 정밀도 |
|---|---|---|---|---|---|
| 압력 저하 (67건) | 로지스틱 회귀, S+H | 35.8% | 26.8% | 1.86 | 6.4% |
| 압력 과다 (68건) | IsolationForest (비지도) | 35.3% (판단 가능 중 51.1%) | 26.4% | 2.51 | 3.3% |
| 누설 (15건) | 엑스트라트리 S+H / 직후 경보 규칙 | 46.7% / 46.7% | 17.8% / 9.6% | 1.79 / 0.45 | 1.6% / 8.4% |

**1. 센서만으로는 무작위 수준이다 (압력 저하).** S만 쓴 지도 모델 5종의 탐지율(9~22%)은 같은 수의 경보를 무작위로 뿌린 값(10~26%)과 비슷하거나 낮다. 행 AUC도 0.52~0.59다.

**2. 고장 이력(H)을 넣으면 나아진다.** 압력 저하 행 AUC가 0.52~0.59에서 0.65~0.82로 오르고, 탐지율도 무작위보다 4~9%p 높다. 다만 오른 부분의 상당수는 **반복 기록**(같은 사건 안 두 번째 이후 기록)을 잡은 것이다. 랜덤포레스트 S+H의 반복 기록 탐지율은 61.5%다.

**3. 직후 경보 규칙은 반복 고장에 매우 강하다.** 학습 없이 "고장이 나면 경보"만 해도 반복 기록을 78~90% 잡고, 오경보는 주 0.4회 정도로 가장 적다. 첫 고장은 거의 못 잡지만(6%), 누설은 다른 고장 직후에 오는 경우가 많아 46.7%를 잡는다. **현장에서 바로 쓸 수 있는 가장 확실한 규칙이다.**

**4. 압력 과다는 지도 모델의 방향이 뒤집힌다.** 지도 모델의 행 AUC가 0.23~0.57로 대부분 0.5보다 낮다. 학습 기간에 배운 패턴이 평가 기간에는 반대로 나타났다는 뜻이다(시간에 따른 변화). 반면 "평소와 다르면 경보"인 IsolationForest는 0.741로 가장 좋았다. 압력 과다는 정답으로 배우기보다 이상 감지가 맞을 수 있다.

**5. 오경보가 목표보다 많다.** 보정 구간에서는 주 1회 이하로 맞췄지만, 평가 구간에서는 주 0.8~2.7회였다. 시간이 지나며 장비 상태가 변해 기준이 어긋난다. 현장에서는 기준을 주기적으로 다시 정해야 한다.

**한계**
- 사건 수가 적다(67·68·15건). 탐지율 30% 근처에서 사건 67건이면 95% 신뢰구간이 약 ±11%p라서, **모델 간 차이 대부분은 우연과 구분되지 않는다.** 누설 15건은 특히 불안정하다.
- 무작위 기대 탐지율은 경보 수만 맞춘 근사값이다(24시간 재경보 금지, 기록 공백은 반영하지 않음).
- 장비마다 시간 분할 지점이 하나뿐이다. 분할 지점을 여러 개로 옮겨 가며 평가하면(walk-forward) 더 안정적이다.
- 하이퍼파라미터는 조정하지 않았다.

**다음 단계 제안**
1. 직후 경보 규칙(반복 고장)과 모델(첫 고장)을 **합친 경보**로 평가한다.
2. 압력 과다는 IsolationForest 같은 이상 감지 쪽으로 따로 다듬는다.
3. 경보 기준을 매주 최근 데이터로 다시 정하는 방식을 시험한다.
